# Kairos-CIDA · Notebook 5: Latent CIDA Training — Zero LLM Memory Overhead

**The LLM is never loaded.** We train Latent CIDA entirely on the cached vectors from Notebook 4,
plus the frozen LM head (saved separately). Total GPU memory: ~4 GB on T4.

### Loss function
```
L_total = L_lang + alpha * L_prm

L_lang  = CrossEntropy(frozen_lm_head(plan_output), answer_ids)
          Language modelling loss on the final answer

L_prm   = ProcessRewardLoss(agent_beliefs_per_round, step_labels)
          Teaches agents to agree on correct steps and disagree on wrong ones
          Prosecutor vs Skeptic disagreement should be HIGH on bad steps
          Consensus disagreement should be LOW on correct steps
```


## 0 · Setup


In [ ]:
import subprocess, sys, os
from getpass import getpass

# 1. Install dependencies
!pip install -q torch transformers datasets accelerate safetensors torchdiffeq

# 2. Securely clone the private repository using a Personal Access Token (PAT)
if not os.path.exists('CIDA-plugin'):
    print('This repository is private. Please enter your GitHub Personal Access Token (PAT):')
    pat = getpass('GitHub PAT: ').strip()
    clone_url = f'https://{pat}@github.com/Kairatzh/CIDA-plugin.git'
    try:
        subprocess.run(['git', 'clone', '--depth', '1', clone_url], check=True)
        print('CIDA-plugin cloned successfully.')
    except Exception as e:
        print('Error cloning repository. Please check your token and try again.', e)
else:
    print('CIDA-plugin already exists.')

sys.path.insert(0, os.path.abspath('CIDA-plugin'))

# 3. Mount Google Drive for persistent caching
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete.')


## 1 · Configuration


In [ ]:
import gc, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tqdm.auto import tqdm
from cida_plugin import LatentCIDAPlugin, LatentConfig

warnings.filterwarnings('ignore')
random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class TrainConfig:
    # Paths
    cache_dir:      str   = '/content/drive/MyDrive/Kairos_Cache'
    weights_in:     str   = '/content/drive/MyDrive/Kairos_CIDA_7B/best'
    weights_out:    str   = '/content/drive/MyDrive/Kairos_CIDA_7B/latent_trained'
    # Architecture
    num_plan_tokens: int  = 8
    # Training
    epochs:          int  = 3
    batch_size:      int  = 16   # no LLM = much larger batch fits on T4
    grad_accum:      int  = 4
    max_lr:          float = 2e-4
    min_lr:          float = 1e-6
    weight_decay:    float = 1e-4
    grad_clip:       float = 1.0
    warmup_steps:    int  = 100
    # Loss weights
    alpha_prm:       float = 0.4   # weight of latent PRM loss
    alpha_orth:      float = 0.05  # weight of orthogonality loss
    # Eval
    eval_every:      int  = 200
    eval_n:          int  = 128

cfg = TrainConfig()
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2 · Load Cached Metadata + Frozen LM Head


In [ ]:
def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

CACHE_DIR = Path(cfg.cache_dir)

with open(str(CACHE_DIR / 'cache_meta.json')) as f:
    meta = json.load(f)

LLM_HIDDEN = meta['llm_hidden']
VOCAB_SIZE  = meta['vocab_size']
print(f'Cache meta: LLM_HIDDEN={LLM_HIDDEN}, VOCAB_SIZE={VOCAB_SIZE}')
print(f'  Train: {meta["n_train"]:,}  |  Val: {meta["n_val"]:,}')

# ── Frozen LM head (the ONLY part of the LLM we need) ────────────────────
# This is a single Linear(LLM_HIDDEN -> VOCAB_SIZE) layer.
# We saved it in Notebook 4. Loading it takes ~30 MB, not 14 GB.
with open(str(CACHE_DIR / 'lm_head_cfg.json')) as f:
    lm_head_cfg = json.load(f)

frozen_lm_head = nn.Linear(
    lm_head_cfg['in_features'],
    lm_head_cfg['out_features'],
    bias=False,
).to(DEVICE)
frozen_lm_head.load_state_dict(
    torch.load(str(CACHE_DIR / 'lm_head.pt'), map_location=DEVICE)
)
for p in frozen_lm_head.parameters(): p.requires_grad = False
frozen_lm_head.eval()

print(f'Frozen LM head loaded: Linear({LLM_HIDDEN} -> {VOCAB_SIZE})')
print(f'Params: {sum(p.numel() for p in frozen_lm_head.parameters())/1e6:.1f}M')
if DEVICE == 'cuda':
    print(f'VRAM so far: {torch.cuda.memory_allocated()/1e9:.3f} GB  (no 7B/8B LLM!)')


## 3 · Load Latent CIDA + PlanProjector


In [ ]:
# Load Latent CIDA from pre-trained weights (from the math/logic training run)
cida_cfg = LatentConfig.from_pretrained(cfg.weights_in)
cida     = LatentCIDAPlugin(cida_cfg)
sd       = torch.load(str(Path(cfg.weights_in)/'pytorch_model.bin'), map_location='cpu')
cida.load_state_dict(sd)
cida     = cida.to(DEVICE).float()

class CIDAPlanProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.p, self.d = num_plan_tokens, d_llm
        d = num_agents * d_cida
        self.net = nn.Sequential(
            nn.Linear(d, d*2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d*2, num_plan_tokens*d_llm),
            nn.LayerNorm(num_plan_tokens*d_llm),
        )
    def forward(self, s):
        B = s.size(0)
        return self.net(s.reshape(B,-1).float()).view(B, self.p, self.d)

plan_proj = CIDAPlanProjector(
    d_cida=cida_cfg.d_hidden, d_llm=LLM_HIDDEN,
    num_agents=cida_cfg.num_agents, num_plan_tokens=cfg.num_plan_tokens,
).to(DEVICE)
plan_proj.load_state_dict(
    torch.load(str(Path(cfg.weights_in)/'plan_projector.pt'), map_location=DEVICE)
)

n_cida = sum(p.numel() for p in cida.parameters())
n_proj = sum(p.numel() for p in plan_proj.parameters())
print(f'Latent CIDA  : {n_cida/1e6:.2f}M params')
print(f'PlanProjector: {n_proj/1e6:.2f}M params')
if DEVICE == 'cuda':
    print(f'VRAM after CIDA+Proj: {torch.cuda.memory_allocated()/1e9:.3f} GB')


## 4 · CachedVectorDataset


In [ ]:
class CachedVectorDataset(Dataset):
    """
    Reads pre-computed LLM hidden states from the Arrow cache.
    No LLM is loaded. Each item returns:
        pooled_hidden  (LLM_HIDDEN,)         CIDA input
        prompt_embeds  (Lq, LLM_HIDDEN)      for plan injection during loss
        answer_ids     (La,)                 CE loss targets
        answer_mask    (La,)                 padding mask
        step_labels    (MAX_STEPS,)          per-step quality signal for PRM loss
    """
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self): return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        return {
            'pooled_hidden': torch.tensor(row['pooled_hidden'], dtype=torch.float32),
            'prompt_embeds': torch.tensor(row['prompt_embeds'], dtype=torch.float32),
            'answer_ids':    torch.tensor(row['answer_ids'],    dtype=torch.long),
            'answer_mask':   torch.tensor(row['answer_mask'],   dtype=torch.long),
            'step_labels':   torch.tensor(row['step_labels'],   dtype=torch.float32),
        }


train_hf = load_from_disk(str(CACHE_DIR / 'train'))
val_hf   = load_from_disk(str(CACHE_DIR / 'val'))

train_ds = CachedVectorDataset(train_hf)
val_ds   = CachedVectorDataset(val_hf)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)

# Shape assertions on one batch
b = next(iter(train_loader))
assert b['pooled_hidden'].shape == (cfg.batch_size, LLM_HIDDEN)
assert b['prompt_embeds'].shape == (cfg.batch_size, b['prompt_embeds'].size(1), LLM_HIDDEN)
assert b['answer_ids'].shape[0]  == cfg.batch_size
assert b['step_labels'].shape    == (cfg.batch_size, 8)
print(f'Dataset shapes verified.')
print(f'  pooled_hidden : {tuple(b["pooled_hidden"].shape)}')
print(f'  prompt_embeds : {tuple(b["prompt_embeds"].shape)}')
print(f'  answer_ids    : {tuple(b["answer_ids"].shape)}')
print(f'  step_labels   : {tuple(b["step_labels"].shape)}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')


## 5 · Latent Process Reward Loss

This is the key innovation in Notebook 5. We teach CIDA agents to **disagree on wrong reasoning steps**
and **converge on correct ones**, using the step-level quality labels from the dataset.

```
step_labels[i] = 1.0  -> correct step  -> agents should AGREE   (low disagreement)
step_labels[i] = 0.0  -> wrong step    -> agents should DISAGREE (high disagreement)
step_labels[i] = 0.5  -> neutral step
```

CIDA runs R rounds of deliberation. We use `b_all[round]` (agent beliefs per round)
to compute per-round disagreement and compare to step labels.
This gives CIDA a **process signal**, not just an outcome signal.


In [ ]:
def latent_prm_loss(
    b_all:       List[torch.Tensor],  # list of (B, M, K) per deliberation round
    step_labels: torch.Tensor,        # (B, MAX_STEPS)  values in [0, 1]
) -> torch.Tensor:
    """
    Latent Process Reward Loss.

    For each deliberation round r, we compute the mean pairwise belief disagreement
    across the 4 agents — this gives a scalar disagreement per round.

    We use step_labels to define the target disagreement:
        target_disagreement = 1 - step_labels[:, r % MAX_STEPS]
        (high label -> low target disagreement, i.e. agents should agree)

    Loss = mean( (actual_disagreement - target_disagreement)^2 )

    This directly trains the multi-agent system to distinguish correct
    from incorrect reasoning steps through the disagreement signal.
    """
    if not b_all:
        return torch.tensor(0.0, device=DEVICE)

    MAX_STEPS = step_labels.size(1)
    total_loss = torch.tensor(0.0, device=DEVICE)

    for r, b_t in enumerate(b_all):
        # b_t : (B, M, K)
        b_t = b_t.to(DEVICE)
        assert b_t.dim() == 3, f'Expected (B, M, K), got {b_t.shape}'

        # Pairwise L1 disagreement across agents
        b_i = b_t.unsqueeze(2)                   # (B, M, 1, K)
        b_j = b_t.unsqueeze(1)                   # (B, 1, M, K)
        pair_dist = (b_i - b_j).abs().sum(-1)    # (B, M, M)

        # Upper triangle (unique pairs)
        M = b_t.size(1)
        mask_upper = torch.triu(torch.ones(M, M, device=DEVICE), diagonal=1).bool()
        actual_disagree = pair_dist[:, mask_upper].mean(dim=1)  # (B,)

        # Target: step label at this round (cycle through MAX_STEPS)
        step_idx = r % MAX_STEPS
        label_r  = step_labels[:, step_idx].to(DEVICE)          # (B,)  in [0,1]
        # High label (correct step) -> target low disagreement
        target_disagree = 1.0 - label_r                         # (B,)

        round_loss = F.mse_loss(actual_disagree, target_disagree)
        total_loss = total_loss + round_loss

    return total_loss / len(b_all)


# Quick shape test
with torch.no_grad():
    dummy_pooled = torch.randn(4, LLM_HIDDEN, device=DEVICE)
    dummy_out    = cida(pooled=dummy_pooled)
    dummy_labels = torch.rand(4, 8)
    test_prm     = latent_prm_loss(dummy_out['b_all'], dummy_labels)
    assert test_prm.ndim == 0, 'PRM loss must be scalar'
print(f'Latent PRM loss test passed. Value: {test_prm.item():.4f}')


## 6 · Forward Pass — No LLM, All Cached


In [ ]:
def cida_forward_cached(
    pooled_hidden: torch.Tensor,   # (B, LLM_HIDDEN)      from cache
    prompt_embeds: torch.Tensor,   # (B, Lq, LLM_HIDDEN)  from cache
    answer_ids:    torch.Tensor,   # (B, La)
    answer_mask:   torch.Tensor,   # (B, La)
    step_labels:   torch.Tensor,   # (B, MAX_STEPS)
) -> Tuple[torch.Tensor, dict]:
    """
    Full Latent CIDA training forward without any LLM forward pass.

    The pooled_hidden and prompt_embeds are READ FROM CACHE.
    Only CIDA + PlanProjector + frozen_lm_head are active.

    Loss decomposition:
        L_lang  = CE(frozen_lm_head(plan_embeds_mean), answer_ids[:, 0])
                  Language loss: plan tokens should predict the first answer token
                  (proxy for answer quality without full LLM generation)
        L_prm   = latent_prm_loss(b_all, step_labels)
                  Process reward: agent disagreement should track step quality
        L_orth  = orthogonality loss from CIDA (representations, not weights)
    """
    B, La = answer_ids.shape
    P     = cfg.num_plan_tokens

    # ── Step 1: CIDA deliberation on cached pooled hidden state ───────────
    cida_out    = cida(pooled=pooled_hidden)             # full dict
    plan_emb    = plan_proj(cida_out['state_final'].s)  # (B, P, LLM_HIDDEN)
    assert plan_emb.shape == (B, P, LLM_HIDDEN), f'plan_emb: {plan_emb.shape}'

    # ── Step 2: Language loss via frozen LM head ──────────────────────────
    # We use mean-pooled plan embeddings as the 'hidden state' for prediction.
    # This is equivalent to asking: does the plan encode the right answer?
    plan_mean   = plan_emb.mean(dim=1)                  # (B, LLM_HIDDEN)
    vocab_logits = frozen_lm_head(plan_mean.to(
        next(frozen_lm_head.parameters()).dtype
    )).float()                                           # (B, VOCAB_SIZE)

    # Target: first non-padding answer token
    first_real   = answer_mask.float().argmax(dim=1)    # (B,)  index of first real token
    target_tok   = answer_ids[torch.arange(B), first_real]  # (B,)
    l_lang       = F.cross_entropy(vocab_logits, target_tok)

    # ── Step 3: Latent PRM loss ───────────────────────────────────────────
    l_prm  = latent_prm_loss(cida_out['b_all'], step_labels)

    # ── Step 4: Orthogonality regulariser (from CIDA's own output) ────────
    l_orth = cida_out.get('L_orth', torch.tensor(0.0, device=DEVICE))
    if l_orth is None: l_orth = torch.tensor(0.0, device=DEVICE)

    # ── Total loss ────────────────────────────────────────────────────────
    loss = l_lang + cfg.alpha_prm * l_prm + cfg.alpha_orth * l_orth

    diag = {
        'l_lang':      l_lang.item(),
        'l_prm':       l_prm.item(),
        'l_orth':      l_orth.item() if isinstance(l_orth, torch.Tensor) else l_orth,
        'disagreement': cida_out['disagreement'].mean().item(),
        'rounds':       cida_out['rounds_used'],
    }
    return loss, diag


print('Forward pass defined (zero LLM overhead).')


## 7 · Optimizer & Scheduler


In [ ]:
trainable = list(cida.parameters()) + list(plan_proj.parameters())
n_train   = sum(p.numel() for p in trainable)

optimizer = torch.optim.AdamW(
    trainable, lr=cfg.max_lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.98)
)

total_steps = (len(train_loader) // cfg.grad_accum) * cfg.epochs

def lr_lambda(step):
    if step < cfg.warmup_steps:
        return (step + 1) / max(cfg.warmup_steps, 1)
    p = (step - cfg.warmup_steps) / max(total_steps - cfg.warmup_steps, 1)
    cos = 0.5 * (1.0 + math.cos(math.pi * p))
    return cfg.min_lr/cfg.max_lr + (1 - cfg.min_lr/cfg.max_lr) * cos

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

print(f'Trainable params : {n_train/1e6:.2f}M')
print(f'Optimizer steps  : {total_steps}')
print(f'Effective batch  : {cfg.batch_size * cfg.grad_accum}')
print(f'Batch size       : {cfg.batch_size}  (no LLM in memory -> can be large!)')


## 8 · Evaluation


In [ ]:
@torch.no_grad()
def evaluate(loader, n=128) -> dict:
    """
    Evaluation computes:
        - Mean loss components
        - Delta: improvement in top-1 vocab prediction vs random baseline
        - Agent disagreement curve across rounds
    """
    cida.eval(); plan_proj.eval()
    metrics = {'l_lang': 0, 'l_prm': 0, 'disagreement': 0,
               'top1_correct': 0, 'total': 0}
    disagree_per_round = {}

    for batch in list(loader)[:n//cfg.batch_size + 1]:
        ph = batch['pooled_hidden'].to(DEVICE)
        pe = batch['prompt_embeds'].to(DEVICE)
        ai = batch['answer_ids'].to(DEVICE)
        am = batch['answer_mask'].to(DEVICE)
        sl = batch['step_labels'].to(DEVICE)

        loss, diag = cida_forward_cached(ph, pe, ai, am, sl)

        # Track per-round disagreement for visualization
        cout = cida(pooled=ph)
        for r, b_t in enumerate(cout['b_all']):
            b_t = b_t.to(DEVICE)
            bi  = b_t.unsqueeze(2); bj = b_t.unsqueeze(1)
            d   = (bi - bj).abs().sum(-1)
            M   = b_t.size(1)
            mask_u = torch.triu(torch.ones(M,M,device=DEVICE), diagonal=1).bool()
            disagree_per_round[r] = disagree_per_round.get(r, 0) + \
                                    d[:, mask_u].mean().item()

        # Top-1 accuracy: does plan_mean predict first answer token?
        plan_emb   = plan_proj(cout['state_final'].s)
        plan_mean  = plan_emb.mean(1)
        logits     = frozen_lm_head(plan_mean.to(
            next(frozen_lm_head.parameters()).dtype)).float()
        first_real = am.float().argmax(1)
        target     = ai[torch.arange(ai.size(0)), first_real]
        correct    = (logits.argmax(-1) == target).sum().item()

        B = ph.size(0)
        metrics['l_lang']      += diag['l_lang'] * B
        metrics['l_prm']       += diag['l_prm'] * B
        metrics['disagreement']+= diag['disagreement'] * B
        metrics['top1_correct']+= correct
        metrics['total']       += B
        if metrics['total'] >= n: break

    T = max(metrics['total'], 1)
    n_batches = max(T // cfg.batch_size, 1)
    return {
        'l_lang':      metrics['l_lang'] / T,
        'l_prm':       metrics['l_prm']  / T,
        'disagreement': metrics['disagreement'] / T,
        'top1_acc':    metrics['top1_correct'] / T,
        'disagree_per_round': {r: v/n_batches for r,v in disagree_per_round.items()},
    }


print('Evaluation function defined.')


## 9 · Training Loop


In [ ]:
Path(cfg.weights_out).mkdir(parents=True, exist_ok=True)

def save_ckpt(tag=''):
    dst = Path(cfg.weights_out) / tag if tag else Path(cfg.weights_out)
    dst.mkdir(parents=True, exist_ok=True)
    cida.save_pretrained(str(dst))
    torch.save(plan_proj.state_dict(), str(dst/'plan_projector.pt'))
    print(f'  Saved -> {dst}')

history = {'step':[], 'l_lang':[], 'l_prm':[], 'disagree':[], 'lr':[]}
val_history = []
global_step = 0
best_lang   = float('inf')

print(f'Training: {cfg.epochs} epochs, {total_steps} optim steps')
print(f'  alpha_prm  = {cfg.alpha_prm}')
print(f'  alpha_orth = {cfg.alpha_orth}')

for epoch in range(cfg.epochs):
    cida.train(); plan_proj.train()
    ep_loss, n_upd = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    t0 = time.time()

    for bi, batch in enumerate(train_loader):
        ph = batch['pooled_hidden'].to(DEVICE)
        pe = batch['prompt_embeds'].to(DEVICE)
        ai = batch['answer_ids'].to(DEVICE)
        am = batch['answer_mask'].to(DEVICE)
        sl = batch['step_labels'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            loss, diag = cida_forward_cached(ph, pe, ai, am, sl)
            loss = loss / cfg.grad_accum

        scaler.scale(loss).backward()
        ep_loss += loss.item() * cfg.grad_accum

        if (bi + 1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1; n_upd += 1

            lr = scheduler.get_last_lr()[0]
            history['step'].append(global_step)
            history['l_lang'].append(diag['l_lang'])
            history['l_prm'].append(diag['l_prm'])
            history['disagree'].append(diag['disagreement'])
            history['lr'].append(lr)

            if global_step % 20 == 0:
                print(f'  E{epoch+1} step {global_step:>4}/{total_steps}'
                      f' | lang={diag["l_lang"]:.4f}'
                      f' prm={diag["l_prm"]:.4f}'
                      f' disag={diag["disagreement"]:.3f}'
                      f' lr={lr:.2e}'
                      f' rounds={diag["rounds"]}')

            if global_step % cfg.eval_every == 0:
                print(f'\n  -- Eval at step {global_step} --')
                val_m = evaluate(val_loader, cfg.eval_n)
                val_history.append({'step': global_step, **val_m})
                print(f'  l_lang={val_m["l_lang"]:.4f}'
                      f'  l_prm={val_m["l_prm"]:.4f}'
                      f'  top1={val_m["top1_acc"]*100:.1f}%'
                      f'  disag={val_m["disagreement"]:.3f}')
                print('  Disagree per round:', {
                    r: round(v,4)
                    for r,v in val_m['disagree_per_round'].items()})
                if val_m['l_lang'] < best_lang:
                    best_lang = val_m['l_lang']
                    save_ckpt('best')
                    print(f'  New best lang loss: {best_lang:.4f}')
                cida.train(); plan_proj.train()
                print()

    print(f'\nEpoch {epoch+1} done in {time.time()-t0:.0f}s'
          f'  avg_loss={ep_loss/max(n_upd*cfg.grad_accum,1):.4f}\n')

save_ckpt('final')
print(f'Training complete. Best lang loss: {best_lang:.4f}')


## 10 · Final Analysis — What Did Agents Learn?


In [ ]:
print('Running final analysis...')
cida.eval(); plan_proj.eval()
final_m = evaluate(val_loader, n=512)

print()
print('='*60)
print('  LATENT TRAINING — FINAL RESULTS')
print('='*60)
print(f'  Language loss   : {final_m["l_lang"]:.4f}')
print(f'  PRM loss        : {final_m["l_prm"]:.4f}')
print(f'  Top-1 acc       : {final_m["top1_acc"]*100:.2f}%')
print(f'  Disagreement    : {final_m["disagreement"]:.4f}')
print()
print('  Disagreement per deliberation round:')
for r, v in sorted(final_m['disagree_per_round'].items()):
    bar = chr(9608) * int(v * 40)
    print(f'    Round {r+1}: {v:.4f}  {bar}')
print()
print('  Expected pattern if training worked:')
print('    Round 1: high disagreement (agents start from priors)')
print('    Round 2: medium (partial consensus)')
print('    Round 3: low   (deliberation converged)')
print()
if history['l_lang']:
    print(f'  Train lang loss: {history["l_lang"][0]:.4f} -> {history["l_lang"][-1]:.4f}')
    print(f'  Train prm loss : {history["l_prm"][0]:.4f} -> {history["l_prm"][-1]:.4f}')
print()
print('='*60)
print(f'  Best checkpoint : {cfg.weights_out}/best')
print(f'  Final checkpoint: {cfg.weights_out}/final')
print('='*60)
